# Week 7 · Notebook 1 — Review & Worked Solutions (Weeks 1–6)
**CSCI 250 — Introduction to Artificial Intelligence · Fall 2026**

Run each section to refresh a week and check your understanding. These are the **worked answers** to the study guide's self-checks and the practice midterm. Use them *after* you have tried the questions yourself.

> Standard data-science stack — preinstalled in Colab. scikit-learn import is guarded so the notebook still runs if it is missing.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import sklearn
    HAS_SK = True
    print('scikit-learn', sklearn.__version__)
except Exception as e:
    HAS_SK = False
    print('scikit-learn missing -> NumPy fallbacks used.', e)

## Week 1 — Python
**Q:** mutable or immutable? **A:** `list` and `dict` are *mutable*; `tuple` and `str` are *immutable*.

In [ ]:
examples = {'list': [1, 2], 'tuple': (1, 2), 'dict': {'a': 1}, 'str': 'hi'}
examples['list'].append(3)          # works (mutable)
try:
    examples['tuple'][0] = 99        # TypeError (immutable)
except TypeError as e:
    print('tuple is immutable:', e)
print(examples['list'])
print({w: len(w) for w in 'the quick brown fox'.split()})  # comprehension

## Week 2 — NumPy & broadcasting
**Q:** shape of `[[1,2,3]] + [[10],[20]]`? **A:** `(1,3)` broadcasts with `(2,1)` to **`(2,3)`**.

In [ ]:
a = np.array([[1, 2, 3]])     # shape (1, 3)
b = np.array([[10], [20]])    # shape (2, 1)
out = a + b                   # broadcasts to (2, 3)
print(out, out.shape)
print('column means (axis=0):', out.mean(axis=0))
print('row means (axis=1):   ', out.mean(axis=1))

## Week 3 — Pandas groupby & a plot
**Q:** what does `df.groupby('city')['sales'].mean()` return? **A:** a `Series` indexed by city, holding each city's mean sales.

In [ ]:
df = pd.DataFrame({
    'city': ['SD', 'SD', 'LA', 'LA', 'LA'],
    'sales': [10, 14, 9, 7, 20],
})
by_city = df.groupby('city')['sales'].mean()
print(by_city)
by_city.plot(kind='bar', title='Mean sales by city'); plt.show()

## Week 4 — train/test split & the sklearn API
**Q:** why hold out a test set? **A:** to estimate **generalization** to unseen data; tuning on the test set leaks information and inflates scores.

In [ ]:
rng = np.random.default_rng(0)
X = rng.standard_normal((100, 1))
y = (3 * X[:, 0] + 0.5 + 0.3 * rng.standard_normal(100))
if HAS_SK:
    from sklearn.model_selection import train_test_split
    from sklearn.linear_model import LinearRegression
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
    model = LinearRegression().fit(Xtr, ytr)     # fit
    print('test R^2:', round(model.score(Xte, yte), 3))  # predict+evaluate
else:
    # NumPy least-squares fallback
    A = np.c_[X, np.ones(len(X))]
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    print('slope, intercept ~', np.round(coef, 3))

## Week 5 — classification metrics from a confusion matrix
**Q:** read a confusion matrix → accuracy/precision/recall.
With `TP=40, FP=10, FN=5, TN=45`:  accuracy=(40+45)/100=**0.85**, precision=40/(40+10)=**0.80**, recall=40/(40+5)≈**0.89**.

In [ ]:
TP, FP, FN, TN = 40, 10, 5, 45
accuracy  = (TP + TN) / (TP + FP + FN + TN)
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
print(f'accuracy={accuracy:.2f}  precision={precision:.2f} '
      f'recall={recall:.2f}  F1={f1:.2f}')
print('Reminder: accuracy misleads on imbalanced data -> prefer precision/recall/F1.')

## Week 5 — overfitting in one picture
**Q:** train acc 0.99, test acc 0.62 — what is wrong? **A:** **overfitting** — the model memorized the training data (incl. noise) and fails to generalize.

In [ ]:
# Fit polynomials of rising degree; watch train error fall but test error rise.
rng = np.random.default_rng(1)
xt = np.sort(rng.uniform(-1, 1, 20)); yt = np.sin(3*xt) + 0.1*rng.standard_normal(20)
xv = np.linspace(-1, 1, 100); yv = np.sin(3*xv)
for deg in [1, 3, 12]:
    c = np.polyfit(xt, yt, deg)
    tr = np.mean((np.polyval(c, xt) - yt)**2)
    te = np.mean((np.polyval(c, xv) - yv)**2)
    print(f'degree {deg:2d}: train MSE={tr:.3f}  test MSE={te:.3f}')
print('High degree: tiny train error, large test error = overfitting.')

## Week 6 — activations & the softmax/next-token idea
**Q:** which activation gives a multi-class probability distribution? **A:** **softmax** (the same step an LLM uses to pick the next token).

In [ ]:
def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

scores = np.array([2.0, 1.0, 0.1])     # raw class scores (logits)
print('relu   :', relu(np.array([-2., 0., 3.])))
print('sigmoid:', np.round(sigmoid(np.array([-2., 0., 2.])), 3))
print('softmax:', np.round(softmax(scores), 3), ' sums to', softmax(scores).sum())

## Week 6 — the gradient-descent update
**Q:** in `w ← w − lr·grad`, what is `lr`? **A:** the **learning rate** — the step size. Too large overshoots/diverges; too small trains slowly.

In [ ]:
# Minimize f(w) = (w - 3)^2  (min at w=3). grad = 2*(w-3).
w, lr = 0.0, 0.1
for step in range(15):
    grad = 2 * (w - 3)
    w = w - lr * grad
print('converged w ~', round(w, 3), '(target 3.0)')

---
# Worked answers to the Practice Midterm (`02_practice_midterm.ipynb`)
Try every item on your own first. These are the worked answers — **no surprises**: every part of the practice midterm has a solution here.

## Practice Midterm — Part A (multiple choice)
- **A1 = (c) tuple** — tuples are immutable; list/dict/set are mutable.
- **A2 = (c) (2,3)** — `(1,3)` broadcasts with `(2,1)` to `(2,3)`.
- **A3 = (b) estimate generalization** — the test set stands in for unseen data.
- **A4 = (b) overfitting** — high train acc, low test acc = memorized noise.
- **A5 = (c) softmax** — turns scores into a multi-class probability distribution.

In [ ]:
# Verify A2 (the broadcast shape) and A5 (softmax sums to 1).
a = np.array([[1, 2, 3]]); b = np.array([[10], [20]])
print('A2 shape:', (a + b).shape)            # (2, 3)
def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()
print('A5 softmax sums to:', round(float(softmax(np.array([2., 1., .1])).sum()), 3))

## Practice Midterm — Part B (short answer)
- **B1.** **AI ⊃ ML ⊃ DL ⊃ GenAI**: AI is the broad goal of intelligent behavior; ML learns patterns from data; DL is ML with multi-layer neural nets; GenAI is DL that *generates* content (text/images/code), e.g. LLMs.
- **B2.** **Regression** predicts a continuous number (metric: MAE or R²); **classification** predicts a discrete label (metric: accuracy or F1).
- **B3.** **Precision** = of the items I flagged positive, how many were actually positive. **Recall** = of the actual positives, how many did I catch.
- **B4.** Without a **nonlinear** activation, stacked linear layers collapse into a single linear layer — no matter how many you stack, you can only draw a straight boundary. Nonlinearity (e.g. ReLU) lets the network bend the boundary into any shape.
- **B5.** **Backprop** computes the gradient of the loss w.r.t. every weight (the chain rule, in one backward pass); the **optimizer** then nudges each weight in the downhill direction (`w ← w − lr·grad`).

## Practice Midterm — Part C (read the code: what does it print?)
Predicted outputs below; run the cell to confirm.

In [ ]:
x = np.arange(6).reshape(2, 3)
print('C1 x.sum(axis=0) ->', x.sum(axis=0).tolist())   # [3, 5, 7]  (down columns)
print('C2 x.sum(axis=1) ->', x.sum(axis=1).tolist())   # [3, 12]    (across rows)
df = pd.DataFrame({'g': ['a', 'a', 'b'], 'v': [1, 3, 10]})
print('C3 groupby mean ->', df.groupby('g')['v'].mean().to_dict())  # {'a': 2.0, 'b': 10.0}
print('C4 f(6) ->', [i*i for i in range(6) if i % 2 == 1])          # [1, 9, 25]

## Practice Midterm — Part D (compute by hand, then verify)
**D1.** With `TP=30, FP=20, FN=10, TN=40`: accuracy=(30+40)/100=**0.70**, precision=30/(30+20)=**0.60**, recall=30/(30+10)=**0.75**, F1=2·0.60·0.75/(0.60+0.75)≈**0.67**.

**D2.** For `f(w)=(w-5)²` with `w0=0`, `lr=0.1`: grad=2·(0−5)=**−10**, so the first update `w ← 0 − 0.1·(−10) = 1.0` moves **toward** 5 (gradient descent always steps downhill).

In [ ]:
# D1
TP, FP, FN, TN = 30, 20, 10, 40
accuracy  = (TP + TN) / (TP + FP + FN + TN)
precision = TP / (TP + FP)
recall    = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)
print(f'D1  accuracy={accuracy:.2f}  precision={precision:.2f} '
      f'recall={recall:.2f}  F1={f1:.2f}')   # 0.70 0.60 0.75 0.67

# D2
w, lr = 0.0, 0.1
grad = 2 * (w - 5)
print('D2  first grad =', grad, ' w after one step =', w - lr * grad, '(toward 5)')

## Practice Midterm — Part E (concept)
**E1 (model answer).** An LLM is a **very deep neural network** (a Transformer) whose behavior is set by billions of **weights** organized into **layers** — the same weights and layers from Week 6, just far more of them. It is trained by **gradient descent** + **backprop** to **minimize cross-entropy** on next-token prediction, and its final **softmax** turns scores over the vocabulary into the probability of each possible next token. Every one of those terms is exactly what you built by hand this half of the course — that is why neural networks are the foundation under LLMs.

## You're ready
If every cell above made sense, you have the Weeks 1–6 core. Now take the **practice midterm** (`02_practice_midterm.ipynb`) on your own — **no AI** — then come back here to check anything you missed.